# RNN, LSTM, GRU — cài đặt bằng NumPy (from scratch) và bằng PyTorch

Notebook này so sánh trực tiếp 3 kiến trúc **RNN, LSTM, GRU**, mỗi kiến
trúc được cài đặt theo **2 cách**:

1. **Thuần NumPy** — tự viết forward pass + backward pass (BPTT) bằng
   tay, không dùng autograd, để hiểu rõ bản chất bên trong.
2. **PyTorch** (`nn.RNN`, `nn.LSTM`, `nn.GRU`) — dùng autograd tự động
   tính đạo hàm, để đối chiếu xem framework "làm hộ" mình những gì.

Dataset: **Tiny Shakespeare** (char-level language model — mạng học dự
đoán ký tự tiếp theo, giống notebook RNN trước đó anh đã học).

Cấu trúc notebook:
- Phần 1: Chuẩn bị dữ liệu (dùng chung cho mọi mô hình)
- Phần 2: NumPy — RNN, LSTM, GRU (lý thuyết + code + train thử)
- Phần 3: PyTorch — RNN, LSTM, GRU (code gọn hơn nhiều nhờ autograd)
- Phần 4: So sánh & ghi chú tổng kết


## Phần 1: Chuẩn bị dữ liệu (dùng chung cho tất cả mô hình)

In [1]:
import numpy as np
import os

DATA_PATH = "tinyshakespeare.txt"  # đổi thành "tinyshakespeare_100k.txt" nếu muốn train nhanh hơn

def load_dataset(path=DATA_PATH):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Không tìm thấy file '{path}'. Anh đặt file Tiny Shakespeare cùng thư mục với notebook này."
        )
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def build_vocab(text):
    chars = sorted(list(set(text)))
    char_to_idx = {ch: i for i, ch in enumerate(chars)}
    idx_to_char = {i: ch for i, ch in enumerate(chars)}
    return chars, char_to_idx, idx_to_char

text = load_dataset()
chars, char_to_idx, idx_to_char = build_vocab(text)
vocab_size = len(chars)

print(f"Dataset có {len(text)} ký tự, vocab có {vocab_size} ký tự khác nhau.")


Dataset có 1115394 ký tự, vocab có 65 ký tự khác nhau.


## Phần 2: NumPy — cài đặt từ đầu (from scratch)

### 2.1 — RNN (vanilla)

Nhắc lại công thức (đã học ở notebook trước):

$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h) \qquad y_t = W_{hy} h_t + b_y$$

Vấn đề chính của RNN thuần: **vanishing gradient** — hidden state đơn
giản dạng $\tanh$ khiến mạng khó nhớ được ngữ cảnh xa. Đây chính là lý
do LSTM/GRU ra đời (dùng thêm các "cổng" — gate — để kiểm soát luồng
thông tin, giữ gradient ổn định hơn qua nhiều bước thời gian).


In [2]:
class RNNNumpy:
    def __init__(self, vocab_size, hidden_size=100, seq_length=25, learning_rate=0.1):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.seq_length = seq_length
        self.learning_rate = learning_rate

        self.Wxh = np.random.randn(hidden_size, vocab_size) * 0.01
        self.Whh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.Why = np.random.randn(vocab_size, hidden_size) * 0.01
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((vocab_size, 1))

        self.params = [self.Wxh, self.Whh, self.Why, self.bh, self.by]
        self.mem = [np.zeros_like(p) for p in self.params]

    def forward_backward(self, inputs, targets, h_prev):
        xs, hs, ys, ps = {}, {}, {}, {}
        hs[-1] = np.copy(h_prev)
        loss = 0.0

        for t in range(len(inputs)):
            xs[t] = np.zeros((self.vocab_size, 1))
            xs[t][inputs[t]] = 1
            hs[t] = np.tanh(self.Wxh @ xs[t] + self.Whh @ hs[t - 1] + self.bh)
            ys[t] = self.Why @ hs[t] + self.by
            exp_y = np.exp(ys[t] - np.max(ys[t]))
            ps[t] = exp_y / np.sum(exp_y)
            loss += -np.log(ps[t][targets[t], 0] + 1e-12)

        dWxh = np.zeros_like(self.Wxh); dWhh = np.zeros_like(self.Whh)
        dWhy = np.zeros_like(self.Why); dbh = np.zeros_like(self.bh); dby = np.zeros_like(self.by)
        dh_next = np.zeros_like(hs[0])

        for t in reversed(range(len(inputs))):
            dy = np.copy(ps[t]); dy[targets[t]] -= 1
            dWhy += dy @ hs[t].T
            dby += dy
            dh = self.Why.T @ dy + dh_next
            dh_raw = (1 - hs[t] * hs[t]) * dh
            dWxh += dh_raw @ xs[t].T
            dWhh += dh_raw @ hs[t - 1].T
            dbh += dh_raw
            dh_next = self.Whh.T @ dh_raw

        grads = [dWxh, dWhh, dWhy, dbh, dby]
        for g in grads:
            np.clip(g, -5, 5, out=g)
        return loss, grads, hs[len(inputs) - 1]

    def update_params(self, grads):
        for p, dp, m in zip(self.params, grads, self.mem):
            m += dp * dp
            p += -self.learning_rate * dp / np.sqrt(m + 1e-8)

    def sample(self, h, seed_idx, n, idx_to_char):
        x = np.zeros((self.vocab_size, 1)); x[seed_idx] = 1
        indices = []
        for _ in range(n):
            h = np.tanh(self.Wxh @ x + self.Whh @ h + self.bh)
            y = self.Why @ h + self.by
            p = np.exp(y - np.max(y)); p = p / np.sum(p)
            idx = np.random.choice(range(self.vocab_size), p=p.ravel())
            x = np.zeros((self.vocab_size, 1)); x[idx] = 1
            indices.append(idx)
        return "".join(idx_to_char[i] for i in indices)


### 2.2 — LSTM (Long Short-Term Memory)

LSTM thêm một "băng chuyền trí nhớ" riêng gọi là **cell state** $c_t$,
được bảo vệ bởi 3 cổng (gate — giá trị 0 đến 1, quyết định "mở/đóng"
luồng thông tin bao nhiêu phần trăm):

- **Forget gate** $f_t$: quên bao nhiêu % trí nhớ cũ $c_{t-1}$
- **Input gate** $i_t$: nhận bao nhiêu % thông tin mới $g_t$ (candidate)
- **Output gate** $o_t$: xuất bao nhiêu % cell state ra thành hidden state

$$z_t = [h_{t-1}; x_t] \quad\text{(nối 2 vector lại)}$$
$$f_t = \sigma(W_f z_t + b_f), \quad i_t = \sigma(W_i z_t + b_i), \quad o_t = \sigma(W_o z_t + b_o)$$
$$g_t = \tanh(W_g z_t + b_g)$$
$$c_t = f_t \odot c_{t-1} + i_t \odot g_t \qquad h_t = o_t \odot \tanh(c_t)$$

($\odot$ = nhân từng phần tử — element-wise, không phải nhân ma trận)

Vì $c_t$ được cộng dồn qua cổng (phép cộng, không phải nhân lặp lại qua
$\tanh$ như RNN thuần), gradient truyền qua $c_t$ ổn định hơn nhiều —
đây là lý do LSTM khắc phục được phần lớn vanishing gradient.


In [ ]:
class LSTMNumpy:
    def __init__(self, vocab_size, hidden_size=100, seq_length=25, learning_rate=0.1):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.seq_length = seq_length
        self.learning_rate = learning_rate
        z_size = hidden_size + vocab_size  # kich thuoc sau khi noi [h_prev, x]

        def init_w(shape):
            return np.random.randn(*shape) * 0.01

        self.Wf = init_w((hidden_size, z_size)); self.bf = np.zeros((hidden_size, 1))
        self.Wi = init_w((hidden_size, z_size)); self.bi = np.zeros((hidden_size, 1))
        self.Wo = init_w((hidden_size, z_size)); self.bo = np.zeros((hidden_size, 1))
        self.Wg = init_w((hidden_size, z_size)); self.bg = np.zeros((hidden_size, 1))
        self.Why = init_w((vocab_size, hidden_size)); self.by = np.zeros((vocab_size, 1))

        self.params = [self.Wf, self.bf, self.Wi, self.bi, self.Wo, self.bo,
                       self.Wg, self.bg, self.Why, self.by]
        self.mem = [np.zeros_like(p) for p in self.params]

    @staticmethod
    def sigmoid(x):
        return 1 / (1 + np.exp(-x))

    def forward_backward(self, inputs, targets, h_prev, c_prev):
        xs, zs, fs, is_, os_, gs, cs, hs, ys, ps = ({} for _ in range(10))
        hs[-1] = np.copy(h_prev)
        cs[-1] = np.copy(c_prev)
        loss = 0.0

        for t in range(len(inputs)):
            xs[t] = np.zeros((self.vocab_size, 1)); xs[t][inputs[t]] = 1
            zs[t] = np.vstack((hs[t-1], xs[t]))  # noi [h_prev, x]

            fs[t] = self.sigmoid(self.Wf @ zs[t] + self.bf)
            is_[t] = self.sigmoid(self.Wi @ zs[t] + self.bi)
            os_[t] = self.sigmoid(self.Wo @ zs[t] + self.bo)
            gs[t] = np.tanh(self.Wg @ zs[t] + self.bg)
            cs[t] = fs[t] * cs[t-1] + is_[t] * gs[t]
            hs[t] = os_[t] * np.tanh(cs[t])

            ys[t] = self.Why @ hs[t] + self.by
            exp_y = np.exp(ys[t] - np.max(ys[t]))
            ps[t] = exp_y / np.sum(exp_y)
            loss += -np.log(ps[t][targets[t], 0] + 1e-12)

        dWf, dbf = np.zeros_like(self.Wf), np.zeros_like(self.bf)
        dWi, dbi = np.zeros_like(self.Wi), np.zeros_like(self.bi)
        dWo, dbo = np.zeros_like(self.Wo), np.zeros_like(self.bo)
        dWg, dbg = np.zeros_like(self.Wg), np.zeros_like(self.bg)
        dWhy, dby = np.zeros_like(self.Why), np.zeros_like(self.by)
        dh_next = np.zeros_like(hs[0])
        dc_next = np.zeros_like(cs[0])

        for t in reversed(range(len(inputs))):
            dy = np.copy(ps[t]); dy[targets[t]] -= 1
            dWhy += dy @ hs[t].T
            dby += dy

            dh = self.Why.T @ dy + dh_next
            do = dh * np.tanh(cs[t])
            do_raw = do * os_[t] * (1 - os_[t])

            dc = dh * os_[t] * (1 - np.tanh(cs[t])**2) + dc_next
            df = dc * cs[t-1]
            df_raw = df * fs[t] * (1 - fs[t])

            di = dc * gs[t]
            di_raw = di * is_[t] * (1 - is_[t])

            dg = dc * is_[t]
            dg_raw = dg * (1 - gs[t]**2)

            dWf += df_raw @ zs[t].T; dbf += df_raw
            dWi += di_raw @ zs[t].T; dbi += di_raw
            dWo += do_raw @ zs[t].T; dbo += do_raw
            dWg += dg_raw @ zs[t].T; dbg += dg_raw

            dz = (self.Wf.T @ df_raw + self.Wi.T @ di_raw +
                  self.Wo.T @ do_raw + self.Wg.T @ dg_raw)
            dh_next = dz[:self.hidden_size, :]
            dc_next = dc * fs[t]

        grads = [dWf, dbf, dWi, dbi, dWo, dbo, dWg, dbg, dWhy, dby]
        for g in grads:
            np.clip(g, -5, 5, out=g)
        return loss, grads, hs[len(inputs)-1], cs[len(inputs)-1]

    def update_params(self, grads):
        for p, dp, m in zip(self.params, grads, self.mem):
            m += dp * dp
            p += -self.learning_rate * dp / np.sqrt(m + 1e-8)

    def sample(self, h, c, seed_idx, n, idx_to_char):
        x = np.zeros((self.vocab_size, 1)); x[seed_idx] = 1
        indices = []
        for _ in range(n):
            z = np.vstack((h, x))
            f = self.sigmoid(self.Wf @ z + self.bf)
            i = self.sigmoid(self.Wi @ z + self.bi)
            o = self.sigmoid(self.Wo @ z + self.bo)
            g = np.tanh(self.Wg @ z + self.bg)
            c = f * c + i * g
            h = o * np.tanh(c)
            y = self.Why @ h + self.by
            p = np.exp(y - np.max(y)); p = p / np.sum(p)
            idx = np.random.choice(range(self.vocab_size), p=p.ravel())
            x = np.zeros((self.vocab_size, 1)); x[idx] = 1
            indices.append(idx)
        return "".join(idx_to_char[i] for i in indices)


### 2.3 — GRU (Gated Recurrent Unit)

GRU đơn giản hơn LSTM — **gộp chung** cell state và hidden state (chỉ
còn 1 trạng thái $h_t$ duy nhất), dùng 2 cổng thay vì 3:

- **Reset gate** $r_t$: quyết định "quên" bao nhiêu hidden state cũ khi
  tính candidate mới
- **Update gate** $z_t$: quyết định giữ lại bao nhiêu $h_{t-1}$ so với
  bao nhiêu candidate mới $\tilde{h}_t$

$$c1_t = [h_{t-1}; x_t]$$
$$r_t = \sigma(W_r \, c1_t + b_r), \qquad z_t = \sigma(W_z \, c1_t + b_z)$$
$$c2_t = [r_t \odot h_{t-1};\, x_t]$$
$$\tilde{h}_t = \tanh(W_h \, c2_t + b_h)$$
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

GRU ít tham số hơn LSTM (2 cổng thay vì 3, không có cell state riêng),
thường train nhanh hơn, hiệu quả gần tương đương LSTM trên nhiều bài
toán — đây là lý do GRU hay được chọn khi cần mô hình gọn nhẹ hơn.


In [ ]:
class GRUNumpy:
    def __init__(self, vocab_size, hidden_size=100, seq_length=25, learning_rate=0.1):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.seq_length = seq_length
        self.learning_rate = learning_rate
        z_size = hidden_size + vocab_size

        def init_w(shape):
            return np.random.randn(*shape) * 0.01

        self.Wr = init_w((hidden_size, z_size)); self.br = np.zeros((hidden_size, 1))
        self.Wz = init_w((hidden_size, z_size)); self.bz = np.zeros((hidden_size, 1))
        self.Wh = init_w((hidden_size, z_size)); self.bh = np.zeros((hidden_size, 1))
        self.Why = init_w((vocab_size, hidden_size)); self.by = np.zeros((vocab_size, 1))

        self.params = [self.Wr, self.br, self.Wz, self.bz, self.Wh, self.bh, self.Why, self.by]
        self.mem = [np.zeros_like(p) for p in self.params]

    @staticmethod
    def sigmoid(x):
        return 1 / (1 + np.exp(-x))

    def forward_backward(self, inputs, targets, h_prev):
        xs, c1s, rs, zgs, c2s, htil, hs, ys, ps = ({} for _ in range(9))
        hs[-1] = np.copy(h_prev)
        loss = 0.0

        for t in range(len(inputs)):
            xs[t] = np.zeros((self.vocab_size, 1)); xs[t][inputs[t]] = 1
            c1s[t] = np.vstack((hs[t-1], xs[t]))
            rs[t] = self.sigmoid(self.Wr @ c1s[t] + self.br)
            zgs[t] = self.sigmoid(self.Wz @ c1s[t] + self.bz)
            c2s[t] = np.vstack((rs[t] * hs[t-1], xs[t]))
            htil[t] = np.tanh(self.Wh @ c2s[t] + self.bh)
            hs[t] = (1 - zgs[t]) * hs[t-1] + zgs[t] * htil[t]

            ys[t] = self.Why @ hs[t] + self.by
            exp_y = np.exp(ys[t] - np.max(ys[t]))
            ps[t] = exp_y / np.sum(exp_y)
            loss += -np.log(ps[t][targets[t], 0] + 1e-12)

        dWr, dbr = np.zeros_like(self.Wr), np.zeros_like(self.br)
        dWz, dbz = np.zeros_like(self.Wz), np.zeros_like(self.bz)
        dWh, dbh = np.zeros_like(self.Wh), np.zeros_like(self.bh)
        dWhy, dby = np.zeros_like(self.Why), np.zeros_like(self.by)
        dh_next = np.zeros_like(hs[0])

        for t in reversed(range(len(inputs))):
            dy = np.copy(ps[t]); dy[targets[t]] -= 1
            dWhy += dy @ hs[t].T
            dby += dy

            dh = self.Why.T @ dy + dh_next

            dzg = dh * (htil[t] - hs[t-1])
            dhtil = dh * zgs[t]
            dh_prev_direct = dh * (1 - zgs[t])

            dhtil_raw = dhtil * (1 - htil[t]**2)
            dWh += dhtil_raw @ c2s[t].T
            dbh += dhtil_raw
            dc2 = self.Wh.T @ dhtil_raw
            dr_hprev = dc2[:self.hidden_size, :]

            dr = dr_hprev * hs[t-1]
            dh_prev_from_r_path = dr_hprev * rs[t]

            dr_raw = dr * rs[t] * (1 - rs[t])
            dWr += dr_raw @ c1s[t].T
            dbr += dr_raw
            dc1_from_r = self.Wr.T @ dr_raw

            dzg_raw = dzg * zgs[t] * (1 - zgs[t])
            dWz += dzg_raw @ c1s[t].T
            dbz += dzg_raw
            dc1_from_z = self.Wz.T @ dzg_raw

            dc1 = dc1_from_r + dc1_from_z
            dh_prev_from_c1 = dc1[:self.hidden_size, :]

            dh_next = dh_prev_direct + dh_prev_from_r_path + dh_prev_from_c1

        grads = [dWr, dbr, dWz, dbz, dWh, dbh, dWhy, dby]
        for g in grads:
            np.clip(g, -5, 5, out=g)
        return loss, grads, hs[len(inputs)-1]

    def update_params(self, grads):
        for p, dp, m in zip(self.params, grads, self.mem):
            m += dp * dp
            p += -self.learning_rate * dp / np.sqrt(m + 1e-8)

    def sample(self, h, seed_idx, n, idx_to_char):
        x = np.zeros((self.vocab_size, 1)); x[seed_idx] = 1
        indices = []
        for _ in range(n):
            c1 = np.vstack((h, x))
            r = self.sigmoid(self.Wr @ c1 + self.br)
            zg = self.sigmoid(self.Wz @ c1 + self.bz)
            c2 = np.vstack((r * h, x))
            htil = np.tanh(self.Wh @ c2 + self.bh)
            h = (1 - zg) * h + zg * htil
            y = self.Why @ h + self.by
            p = np.exp(y - np.max(y)); p = p / np.sum(p)
            idx = np.random.choice(range(self.vocab_size), p=p.ravel())
            x = np.zeros((self.vocab_size, 1)); x[idx] = 1
            indices.append(idx)
        return "".join(idx_to_char[i] for i in indices)


### 2.4 — Hàm train dùng chung + chạy thử cả 3 model NumPy

Vì RNN chỉ cần truyền `h_prev`, còn LSTM cần thêm `c_prev`, hàm train
viết chung bằng cách nhận vào 1 tham số `cell_type` để biết cách gọi
`forward_backward` và `sample` cho đúng.


In [ ]:
def train_numpy_model(model, cell_type, text, char_to_idx, idx_to_char,
                       hidden_size, n_iterations=2000, print_every=500, sample_len=150):
    pointer = 0
    h_prev = np.zeros((hidden_size, 1))
    c_prev = np.zeros((hidden_size, 1)) if cell_type == "lstm" else None
    smooth_loss = -np.log(1.0 / model.vocab_size) * model.seq_length

    for iteration in range(n_iterations):
        if pointer + model.seq_length + 1 >= len(text) or iteration == 0:
            h_prev = np.zeros((hidden_size, 1))
            if cell_type == "lstm":
                c_prev = np.zeros((hidden_size, 1))
            pointer = 0

        inputs = [char_to_idx[ch] for ch in text[pointer: pointer + model.seq_length]]
        targets = [char_to_idx[ch] for ch in text[pointer + 1: pointer + model.seq_length + 1]]

        if cell_type == "lstm":
            loss, grads, h_prev, c_prev = model.forward_backward(inputs, targets, h_prev, c_prev)
        else:
            loss, grads, h_prev = model.forward_backward(inputs, targets, h_prev)
        model.update_params(grads)

        smooth_loss = smooth_loss * 0.999 + loss * 0.001

        if iteration % print_every == 0:
            if cell_type == "lstm":
                sample_text = model.sample(h_prev, c_prev, inputs[0], sample_len, idx_to_char)
            else:
                sample_text = model.sample(h_prev, inputs[0], sample_len, idx_to_char)
            print(f"[{cell_type.upper()}] iter {iteration}, smooth loss: {smooth_loss:.3f}")
            print(sample_text)
            print("-" * 60)

        pointer += model.seq_length

    return model


In [ ]:
# Chạy thử nhanh cả 3 model NumPy (số vòng lặp nhỏ để demo — anh tăng
# n_iterations lên vài chục nghìn nếu muốn kết quả giống văn bản thật hơn)
hidden_size = 128
seq_length = 25
n_iterations_demo = 2000  # tăng lên 20000+ để có kết quả tốt hơn (sẽ chạy lâu hơn)

np.random.seed(1)
rnn_model = RNNNumpy(vocab_size, hidden_size, seq_length, learning_rate=0.1)
train_numpy_model(rnn_model, "rnn", text, char_to_idx, idx_to_char, hidden_size, n_iterations_demo)


In [ ]:
np.random.seed(1)
lstm_model = LSTMNumpy(vocab_size, hidden_size, seq_length, learning_rate=0.1)
train_numpy_model(lstm_model, "lstm", text, char_to_idx, idx_to_char, hidden_size, n_iterations_demo)


In [ ]:
np.random.seed(1)
gru_model = GRUNumpy(vocab_size, hidden_size, seq_length, learning_rate=0.1)
train_numpy_model(gru_model, "gru", text, char_to_idx, idx_to_char, hidden_size, n_iterations_demo)


## Phần 3: PyTorch — RNN, LSTM, GRU (dùng autograd)

Điểm khác biệt lớn nhất so với NumPy: **không cần tự viết backward**.
`loss.backward()` tự tính toàn bộ gradient (kể cả BPTT qua nhiều bước
thời gian), `optimizer.step()` tự cập nhật tham số.

Một model class duy nhất, chọn `cell_type` để dùng `nn.RNN`, `nn.LSTM`,
hay `nn.GRU` — vì PyTorch đã cài đặt sẵn cả 3, không cần viết tay công
thức forward/backward như ở Phần 2.


In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Đang dùng thiết bị:", device)

class CharRNNTorch(nn.Module):
    def __init__(self, vocab_size, hidden_size=128, cell_type="lstm", num_layers=1):
        super().__init__()
        self.cell_type = cell_type
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)

        if cell_type == "rnn":
            self.rnn = nn.RNN(hidden_size, hidden_size, num_layers=num_layers, batch_first=True)
        elif cell_type == "lstm":
            self.rnn = nn.LSTM(hidden_size, hidden_size, num_layers=num_layers, batch_first=True)
        elif cell_type == "gru":
            self.rnn = nn.GRU(hidden_size, hidden_size, num_layers=num_layers, batch_first=True)
        else:
            raise ValueError("cell_type phai la 'rnn', 'lstm', hoac 'gru'")

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        emb = self.embedding(x)          # (batch, seq_len, hidden_size)
        out, hidden = self.rnn(emb, hidden)
        logits = self.fc(out)            # (batch, seq_len, vocab_size)
        return logits, hidden


In [ ]:
data_idx = [char_to_idx[c] for c in text]

def get_batch(seq_length=25, batch_size=32):
    starts = torch.randint(0, len(data_idx) - seq_length - 1, (batch_size,))
    xb = torch.stack([torch.tensor(data_idx[s:s+seq_length]) for s in starts])
    yb = torch.stack([torch.tensor(data_idx[s+1:s+seq_length+1]) for s in starts])
    return xb.to(device), yb.to(device)

def sample_torch(model, seed_char, n, char_to_idx, idx_to_char, device):
    model.eval()
    idx = char_to_idx[seed_char]
    x = torch.tensor([[idx]], dtype=torch.long).to(device)
    hidden = None
    result = [seed_char]
    with torch.no_grad():
        for _ in range(n):
            logits, hidden = model(x, hidden)
            probs = torch.softmax(logits[0, -1], dim=-1)
            idx = torch.multinomial(probs, 1).item()
            result.append(idx_to_char[idx])
            x = torch.tensor([[idx]], dtype=torch.long).to(device)
    model.train()
    return "".join(result)

def train_torch_model(cell_type, n_iterations=1000, print_every=200,
                       hidden_size=128, seq_length=25, batch_size=32, lr=0.005):
    model = CharRNNTorch(vocab_size, hidden_size, cell_type=cell_type).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for it in range(n_iterations):
        xb, yb = get_batch(seq_length, batch_size)
        logits, _ = model(xb)
        loss = criterion(logits.reshape(-1, vocab_size), yb.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if it % print_every == 0:
            sample_text = sample_torch(model, text[0], 150, char_to_idx, idx_to_char, device)
            print(f"[{cell_type.upper()}] iter {it}, loss: {loss.item():.3f}")
            print(sample_text)
            print("-" * 60)

    return model


In [ ]:
# Số vòng lặp demo nhỏ để chạy nhanh — tăng lên vài nghìn nếu muốn văn bản mượt hơn
n_iterations_demo_torch = 1000

torch_rnn = train_torch_model("rnn", n_iterations=n_iterations_demo_torch)


In [ ]:
torch_lstm = train_torch_model("lstm", n_iterations=n_iterations_demo_torch)


In [ ]:
torch_gru = train_torch_model("gru", n_iterations=n_iterations_demo_torch)


## Phần 4: So sánh & ghi chú tổng kết

| | NumPy (from scratch) | PyTorch (`nn.RNN/LSTM/GRU`) |
|---|---|---|
| Backward | Tự viết BPTT bằng tay | `loss.backward()` tự động (autograd) |
| Tốc độ code | Chậm hơn nhiều (thuần Python/NumPy, 1 chuỗi/lần) | Nhanh hơn nhiều (tối ưu C++/CUDA, xử lý theo batch) |
| Mục đích | Hiểu bản chất toán học bên trong | Dùng thực tế, hiệu năng cao |
| Độ dài code | Dài (phải viết forward + backward cho từng cổng) | Ngắn gọn (chỉ cần khai báo `nn.LSTM(...)`) |

**Ghi nhớ về sự khác biệt giữa 3 kiến trúc (áp dụng cho cả 2 cách cài đặt):**

- **RNN**: đơn giản nhất, nhưng dễ bị vanishing gradient — khó nhớ ngữ
  cảnh xa.
- **LSTM**: có cell state $c_t$ riêng + 3 cổng (forget, input, output) —
  khắc phục vanishing gradient tốt, nhưng nhiều tham số nhất, chậm nhất.
- **GRU**: gộp hidden state + cell state làm một, chỉ 2 cổng (reset,
  update) — ít tham số hơn LSTM, train nhanh hơn, hiệu quả thường gần
  tương đương LSTM.

**Gợi ý thử nghiệm tiếp theo:**
- Tăng `n_iterations` lên vài chục nghìn để thấy văn bản sinh ra mượt
  hơn nhiều so với demo ngắn ở trên.
- Thử `num_layers=2` trong bản PyTorch để xem stacked RNN/LSTM/GRU có
  cải thiện gì không.
- So sánh thời gian chạy thực tế giữa bản NumPy và PyTorch với cùng số
  vòng lặp — sẽ thấy rõ vì sao framework tối ưu quan trọng khi làm việc
  thực tế, dù bản NumPy giúp hiểu sâu hơn về mặt lý thuyết.
